# Phase 2: Data Cleaning & EDA
### Digital Wallet Cohort & Retention Project
**Input:** `data/raw/` | **Output:** `data/cleaned/`

In [3]:
import pandas as pd
import numpy as np

INPUT_DIR  = "../data/raw"
OUTPUT_DIR = "../data/cleaned"

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)

## 1. Initial Data Inspection

In [4]:
users = pd.read_csv(f"{INPUT_DIR}/users_dim.csv")
logs  = pd.read_csv(f"{INPUT_DIR}/user_engagement_logs.csv")
txns  = pd.read_csv(f"{INPUT_DIR}/transactions_fact.csv")

users["signup_timestamp"]      = pd.to_datetime(users["signup_timestamp"])
logs["event_timestamp"]        = pd.to_datetime(logs["event_timestamp"])
txns["transaction_timestamp"]  = pd.to_datetime(txns["transaction_timestamp"])

In [5]:
print("--- DataFrame Shapes ---")
print(f"users_dim:            {users.shape}")
print(f"user_engagement_logs: {logs.shape}")
print(f"transactions_fact:    {txns.shape}")

--- DataFrame Shapes ---
users_dim:            (25000, 6)
user_engagement_logs: (169340, 4)
transactions_fact:    (45018, 6)


In [6]:
print("--- Missing Values: users_dim ---")
print(users.isnull().sum())

--- Missing Values: users_dim ---
user_id                   0
signup_timestamp          0
signup_channel            0
device_os               737
age                       0
onboarding_completed      0
dtype: int64


In [7]:
print("--- Missing Values: user_engagement_logs ---")
print(logs.isnull().sum())

--- Missing Values: user_engagement_logs ---
event_id           0
user_id            0
event_timestamp    0
event_name         0
dtype: int64


In [8]:
print("--- Missing Values: transactions_fact ---")
print(txns.isnull().sum())

--- Missing Values: transactions_fact ---
transaction_id              0
user_id                     0
transaction_timestamp       0
transaction_type            0
amount                   1307
status                      0
dtype: int64


## 2. Data Cleaning & Anomaly Handling

### 2a. Deduplication

In [9]:
dupe_count = txns.duplicated(subset="transaction_id", keep="first").sum()
print(f"Duplicate transaction_id rows found: {dupe_count}")

txns = txns.drop_duplicates(subset="transaction_id", keep="first")
print(f"Shape after dedup: {txns.shape}")

Duplicate transaction_id rows found: 665
Shape after dedup: (44353, 6)


### 2b. Missing Values

In [10]:
# device_os: impute with Unknown — missing platform does not
# invalidate the rest of the user record
missing_device_os = users["device_os"].isnull().sum()
users["device_os"] = users["device_os"].fillna("Unknown")
print(f"Imputed {missing_device_os} missing device_os values with 'Unknown'.")

Imputed 737 missing device_os values with 'Unknown'.


In [11]:
# amount: drop rows entirely — we cannot evaluate financial volume
# or monetization metrics with a missing currency field
missing_amount = txns["amount"].isnull().sum()
txns = txns.dropna(subset=["amount"])
print(f"Dropped {missing_amount} rows with missing amount.")
print(f"Shape after amount cleanup: {txns.shape}")

Dropped 1286 rows with missing amount.
Shape after amount cleanup: (43067, 6)


### 2c. Temporal Logic Verification

In [12]:
# Remove events that were logged before the user even signed up
# This is caused by a simulated system timezone bug in the raw data
signup_lookup  = users.set_index("user_id")["signup_timestamp"]

logs_signup_ts = logs["user_id"].map(signup_lookup)
bad_logs_mask  = logs["event_timestamp"] < logs_signup_ts
bad_logs_count = bad_logs_mask.sum()
logs           = logs.loc[~bad_logs_mask].reset_index(drop=True)
print(f"Removed {bad_logs_count} log rows where event predates signup.")

Removed 0 log rows where event predates signup.


In [13]:
# Same check for transactions
txns_signup_ts = txns["user_id"].map(signup_lookup)
bad_txns_mask  = txns["transaction_timestamp"] < txns_signup_ts
bad_txns_count = bad_txns_mask.sum()
txns           = txns.loc[~bad_txns_mask].reset_index(drop=True)
print(f"Removed {bad_txns_count} transaction rows where timestamp predates signup.")
print(f"\nFinal shapes -> users: {users.shape} | logs: {logs.shape} | txns: {txns.shape}")

Removed 44 transaction rows where timestamp predates signup.

Final shapes -> users: (25000, 6) | logs: (169340, 4) | txns: (43023, 6)


## 3. Baseline Metrics (EDA)

### 3a. Core Activation Metric — Onboarding Conversion Rate

In [14]:
rate = users["onboarding_completed"].mean()
print(f"Overall Onboarding Conversion Rate: {rate:.2%}")

Overall Onboarding Conversion Rate: 75.83%


### 3b. Platform Reliability Metric — Transaction Success Rate

In [15]:
success_rate = (txns["status"] == "Success").mean()
print(f"Overall Transaction Success Rate: {success_rate:.2%}")

Overall Transaction Success Rate: 90.22%


### 3c. Acquisition Performance by Channel

In [16]:
channel_performance = users.groupby("signup_channel").agg(
    total_signups   = ("user_id", "count"),
    onboarding_rate = ("onboarding_completed", "mean"),
).round(4)

print("Acquisition Performance by signup_channel:")
channel_performance

Acquisition Performance by signup_channel:


,total_signups,onboarding_rate
signup_channel,,
Organic,9969,0.8138
Paid Ads,11308,0.6932
Referral,3723,0.8071


### 3d. Average Transaction Amount by Channel

In [17]:
txns_with_channel = txns.merge(
    users[["user_id", "signup_channel"]],
    on  = "user_id",
    how = "left"
)

avg_amount = (
    txns_with_channel
    .groupby("signup_channel")["amount"]
    .mean()
    .round(2)
    .rename("avg_transaction_amount")
)

print("Average Transaction Amount by signup_channel:")
avg_amount

Average Transaction Amount by signup_channel:


signup_channel
Organic     172.04
Paid Ads    169.63
Referral    170.90
Name: avg_transaction_amount, dtype: float64

## 4. Root Cause Isolation — March 2026 Anomaly

### 4a. Build Monthly Cohort Labels

In [18]:
users["signup_cohort"] = users["signup_timestamp"].dt.to_period("M").astype(str)

cohort_onboarding = users.groupby("signup_cohort")["onboarding_completed"].agg(
    total_signups   = "count",
    onboarding_rate = "mean"
).round(4)

print("Onboarding completion rate by signup cohort:")
cohort_onboarding

Onboarding completion rate by signup cohort:


,total_signups,onboarding_rate
signup_cohort,,
2025-01,671,0.7466
2025-02,677,0.7415
2025-03,891,0.7475
2025-04,867,0.7693
2025-05,984,0.7429
2025-06,1059,0.7677
2025-07,1208,0.7566
2025-08,1251,0.7570
2025-09,1345,0.7599


### 4b. Feb 2026 vs March 2026 Direct Comparison

In [19]:
feb_rate  = cohort_onboarding.loc["2026-02", "onboarding_rate"]
mar_rate  = cohort_onboarding.loc["2026-03", "onboarding_rate"]
rate_drop = feb_rate - mar_rate

print("--- Feb 2026 vs March 2026 Onboarding Comparison ---")
print(f"February 2026 onboarding rate : {feb_rate:.2%}")
print(f"March 2026 onboarding rate    : {mar_rate:.2%}")
print(f"Absolute drop                 : {rate_drop:.2%}")
print(f"Relative decline              : {rate_drop / feb_rate:.1%}")

--- Feb 2026 vs March 2026 Onboarding Comparison ---
February 2026 onboarding rate : 75.71%
March 2026 onboarding rate    : 76.46%
Absolute drop                 : -0.75%
Relative decline              : -1.0%


### 4c. Friction Event Frequency Comparison

In [20]:
march_users = users.loc[users["signup_cohort"] == "2026-03", "user_id"]
feb_users   = users.loc[users["signup_cohort"] == "2026-02", "user_id"]

march_logs  = logs[logs["user_id"].isin(march_users)]
feb_logs    = logs[logs["user_id"].isin(feb_users)]

march_dist  = march_logs["event_name"].value_counts(normalize=True).round(4)
feb_dist    = feb_logs["event_name"].value_counts(normalize=True).round(4)

event_comparison = pd.DataFrame({
    "march_2026_pct" : march_dist,
    "feb_2026_pct"   : feb_dist,
}).fillna(0).sort_values("march_2026_pct", ascending=False)

print("Event distribution: March 2026 vs February 2026:")
event_comparison

Event distribution: March 2026 vs February 2026:


,march_2026_pct,feb_2026_pct
event_name,,
App_Open,0.5056,0.5276
Onboarding_Step_1,0.1548,0.1472
Onboarding_Completed,0.1184,0.1115
View_Dashboard,0.1004,0.0956
Link_Bank_Account,0.0642,0.0618
Initiate_Transfer,0.0458,0.0438
KYC_Failed,0.0062,0.0067
Support_Ticket_Opened,0.0046,0.0058


In [21]:
friction_events = ["KYC_Failed", "Support_Ticket_Opened"]
print("--- Friction Event Spotlight ---")
for ev in friction_events:
    m = march_dist.get(ev, 0)
    f = feb_dist.get(ev, 0)
    print(f"{ev:>25}: March = {m:.2%}  |  Feb = {f:.2%}")

--- Friction Event Spotlight ---
               KYC_Failed: March = 0.62%  |  Feb = 0.67%
    Support_Ticket_Opened: March = 0.46%  |  Feb = 0.58%


**Finding:** Onboarding completion and friction event rates are essentially
flat between February and March 2026. The March drop is not happening at the
onboarding stage — it surfaces downstream in Month 1+ retention and transaction
behavior, which the Phase 3 SQL cohort matrix will expose directly.

## 5. Export Cleaned Datasets

In [22]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

users_export = users.drop(columns=["signup_cohort"])

users_export.to_csv(f"{OUTPUT_DIR}/cleaned_users_dim.csv",            index=False)
logs.to_csv(f"{OUTPUT_DIR}/cleaned_user_engagement_logs.csv",          index=False)
txns.to_csv(f"{OUTPUT_DIR}/cleaned_transactions_fact.csv",             index=False)

print("Exported successfully:")
print(f"  cleaned_users_dim.csv            -> {users_export.shape[0]:,} rows")
print(f"  cleaned_user_engagement_logs.csv -> {logs.shape[0]:,} rows")
print(f"  cleaned_transactions_fact.csv    -> {txns.shape[0]:,} rows")
print("\nPhase 2 complete.")

Exported successfully:
  cleaned_users_dim.csv            -> 25,000 rows
  cleaned_user_engagement_logs.csv -> 169,340 rows
  cleaned_transactions_fact.csv    -> 43,023 rows

Phase 2 complete.
